>Feature engineering transforms raw transaction data into behavioral signals that help reveal abnormal patterns. In anomaly detection, the quality of features often matters more than the choice of model.
>Effective fraud detection focuses on behavioral rather than static features, with time playing a central role in capturing changes in activity. At the same time, care must be taken to avoid information leakage, as it can lead to misleading model performance.

# PaySim

## Time-based Transaction Count Features

In [ ]:
import pandas as pd

paysim = pd.read_csv("../data/raw/paysim.csv")

In [ ]:
# sort step feature
paysim = paysim.sort_values(["nameOrig", "step"]).reset_index(drop=True)

In [6]:
paysim.groupby("nameOrig")["step"].is_monotonic_increasing.all()

np.True_

In [7]:
import numpy as np

groups = paysim.groupby("nameOrig").indices
steps_all = paysim["step"].values

windows = {
    "24h": 24,
    "72h": 72,
    "7d": 168
}

for name, window in windows.items():
    tx_count = np.zeros(len(paysim), dtype=np.int32)

    for idx in groups.values():
        steps = steps_all[idx]
        j = 0

        for i in range(len(steps)):
            while steps[j] < steps[i] - window:
                j += 1
            tx_count[idx[i]] = i - j + 1

    paysim[f"tx_count_last_{name}"] = tx_count


In [8]:
paysim["tx_count_last_24h"].isna().sum()

np.int64(0)

In [9]:
paysim["tx_count_last_24h"].describe()


,tx_count_last_24h
count,6.362620e+06
mean,1.000181e+00
std,1.345454e-02
min,1.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,2.000000e+00


In [13]:
paysim["tx_count_last_72h"].isna().sum()

np.int64(0)

In [12]:
paysim["tx_count_last_72h"].describe()

,tx_count_last_72h
count,6.362620e+06
mean,1.000440e+00
std,2.096200e-02
min,1.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,2.000000e+00


In [10]:
paysim["tx_count_last_7d"].isna().sum()

np.int64(0)

In [11]:
paysim["tx_count_last_7d"].describe()


,tx_count_last_7d
count,6.362620e+06
mean,1.000882e+00
std,2.970701e-02
min,1.000000e+00
25%,1.000000e+00
50%,1.000000e+00
75%,1.000000e+00
max,3.000000e+00


In [14]:
paysim["tx_count_total_user"] = (
    paysim
    .groupby("nameOrig")["step"]
    .transform("count")
)

To capture short-term and mid-term behavioral patterns, several time-based transaction count features were engineered using sliding temporal windows. Specifically, the number of transactions performed by each user within the last 24 hours, 72 hours, and 7 days was computed based on the step variable, which represents elapsed hours in the PaySim simulation. These features aim to model bursty or unusually frequent transaction behavior, which is often indicative of fraudulent activity.

In addition, a global baseline feature (tx_count_total_user) was introduced to represent the total number of transactions performed by each user throughout the entire simulation. This feature provides contextual information about a user's overall activity level and allows the model to distinguish between inherently low-activity users and sudden deviations from typical behavior when combined with time-based counts.

## Time-based Transaction Amount Features

In [16]:
amounts_all = paysim["amount"].values

In [21]:
windows = {
    "24h": 24,
    "7d": 168
}

for name, window in windows.items():
    sum_amount = np.zeros(len(paysim), dtype=np.float32)

    for idx in groups.values():
        steps = steps_all[idx]
        amounts = amounts_all[idx]

        j = 0
        running_sum = 0.0

        for i in range(len(steps)):
            running_sum += amounts[i]

            while steps[j] < steps[i] - window:
                running_sum -= amounts[j]
                j += 1

            sum_amount[idx[i]] = running_sum

    paysim[f"sum_amount_last_{name}"] = sum_amount
    paysim[f"avg_amount_last_{name}"] = (
        sum_amount / paysim[f"tx_count_last_{name}"]
    )

In [19]:
paysim[[
    "sum_amount_last_24h",
    "avg_amount_last_24h"
]].isna().sum()


,0
sum_amount_last_24h,0
avg_amount_last_24h,0


In [20]:
paysim[[
    "sum_amount_last_24h",
    "avg_amount_last_24h"
]].describe()


,sum_amount_last_24h,avg_amount_last_24h
count,6.362620e+06,6.362620e+06
mean,1.799040e+05,1.798675e+05
std,5.947048e+05,5.944260e+05
min,0.000000e+00,0.000000e+00
25%,1.339283e+04,1.339203e+04
50%,7.489770e+04,7.488427e+04
75%,2.087495e+05,2.087176e+05
max,9.244552e+07,9.244552e+07


In [22]:
paysim[[
    "sum_amount_last_7d",
    "avg_amount_last_7d"
]].isna().sum()


,0
sum_amount_last_7d,0
avg_amount_last_7d,0


In [23]:
paysim[[
    "sum_amount_last_7d",
    "avg_amount_last_7d"
]].describe()


,sum_amount_last_7d,avg_amount_last_7d
count,6.362620e+06,6.362620e+06
mean,1.800258e+05,1.798618e+05
std,5.950166e+05,5.942920e+05
min,0.000000e+00,0.000000e+00
25%,1.340401e+04,1.339981e+04
50%,7.498725e+04,7.492885e+04
75%,2.088801e+05,2.087044e+05
max,9.244552e+07,9.244552e+07


To capture spending behavior over different time scales, time-based transaction amount features were created using sliding windows aligned with the PaySim simulation time. For each transaction, both the total and average transferred amounts were calculated per user over the past 24 hours and the past 7 days.

The 24-hour window is intended to capture short-term spending spikes that may signal sudden abnormal behavior, while the 7-day window provides a broader view of recent activity. The similar distribution observed across these features reflect the sparse transaction patterns in PaySim, highlighting the need to combine amount-based features with frequency-based singals for effective anomaly detection.